# THUMOS-14 Diving → MIL Bags with ViT-B/16 Encoding

Binary classification: **Diving** vs **no Diving**.  
Each video is split so that every bag contains at most one action segment.  
Saves bag-level labels (0/1) and instance-level labels per frame.

In [ ]:
import json, os, numpy as np, torch, torchvision, cv2
from torchvision.models import vit_b_16, ViT_B_16_Weights
import os
Base_dir = "." # "/cluster/tufts/hugheslab/datasets/thumos-14/"
ANNO_PATH = os.path.join(Base_dir,"thumos-14/annotations/thumos_14_anno.json")
FC_PATH   = os.path.join(Base_dir,"thumos-14/annotations/frame_count_raw_video.json")
VIDEO_DIR = os.path.join(Base_dir,"thumos-14/video")
OUT_DIR   = os.path.join(Base_dir,"thumos-14/encoded_thumos_diving")
SAMPLE_FPS = 1          # sample 1 frame per second to keep it manageable
ACTION    = "Diving"
os.makedirs(OUT_DIR, exist_ok=True)

anno = json.load(open(ANNO_PATH))["database"]
fc   = json.load(open(FC_PATH))

def read_video_frames(path):
    """Read all frames from a video file, returns (N, H, W, C) uint8 tensor (RGB)."""
    cap = cv2.VideoCapture(path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame[:, :, ::-1].copy())  # BGR → RGB
    cap.release()
    return torch.from_numpy(np.stack(frames))

## 1. Build frame-level labels & split into bags (step by step)

In [ ]:
# Pick one video to walk through the logic
vid = "video_validation_0000161"
info, fps = anno[vid], fc[vid]["video_fps"]
n_frames = fc[vid]["total_frames"]
step = int(fps / SAMPLE_FPS)
sampled_indices = list(range(0, n_frames, step))

action_segs = sorted(
    [a["segment"] for a in info["annotations"] if a["label"] == ACTION],
    key=lambda s: s[0],
)
print(f"{vid}: {n_frames} frames @ {fps}fps, duration {fc[vid]['video_seconds']:.1f}s")
print(f"Sampled at {SAMPLE_FPS} FPS -> {len(sampled_indices)} frames")
print(f"Diving segments (seconds): {action_segs}")

video_validation_0000161: 1852 frames @ 30fps, duration 61.7s
Sampled at 1 FPS → 62 frames
Diving segments (seconds): [[11.3, 14.1], [21.0, 23.9], [26.4, 29.4], [50.9, 55.4]]


In [3]:
# Step 1: Create frame-level binary labels from the time segments
labels = np.zeros(len(sampled_indices), dtype=np.int64)
for seg_start, seg_end in action_segs:
    for i, fi in enumerate(sampled_indices):
        if seg_start <= fi / fps <= seg_end:
            labels[i] = 1

print("Frame-level labels (1=Diving, 0=not):")
print(labels.tolist())

Frame-level labels (1=Diving, 0=not):
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]


In [4]:
# Step 2: Find where each contiguous action block starts and ends
diff = np.diff(labels, prepend=0, append=0)
starts = np.where(diff == 1)[0]
ends   = np.where(diff == -1)[0]

print(f"Found {len(starts)} action blocks:")
for s, e in zip(starts, ends):
    print(f"  frames [{s}:{e}] → times ~{s}s to ~{e}s")

Found 4 action blocks:
  frames [12:15] → times ~12s to ~15s
  frames [21:24] → times ~21s to ~24s
  frames [27:30] → times ~27s to ~30s
  frames [51:56] → times ~51s to ~56s


In [5]:
# Step 3: Split at midpoints between consecutive action blocks → one action per bag
boundaries = [0]
for i in range(len(starts) - 1):
    mid = (ends[i] + starts[i + 1]) // 2
    boundaries.append(mid)
boundaries.append(len(labels))

print(f"Split boundaries: {boundaries}\n")
for i in range(len(boundaries) - 1):
    s, e = boundaries[i], boundaries[i + 1]
    seg = labels[s:e]
    print(f"Bag {i} [{s}:{e}] ({e-s} frames)  bag_label={seg.max()}")
    print(f"  instance_labels = {seg.tolist()}\n")

Split boundaries: [0, np.int64(18), np.int64(25), np.int64(40), 62]

Bag 0 [0:18] (18 frames)  bag_label=1
  instance_labels = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0]

Bag 1 [18:25] (7 frames)  bag_label=1
  instance_labels = [0, 0, 0, 1, 1, 1, 0]

Bag 2 [25:40] (15 frames)  bag_label=1
  instance_labels = [0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Bag 3 [40:62] (22 frames)  bag_label=1
  instance_labels = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]



### Now apply to all videos

In [6]:
bags = []  # (video_id, sampled_frame_indices, bag_label, instance_labels)

for vid, info in anno.items():
    if vid not in fc:
        continue
    fps = fc[vid]["video_fps"]
    n_frames = fc[vid]["total_frames"]
    step = int(fps / SAMPLE_FPS)
    sampled_indices = list(range(0, n_frames, step))
    n_sampled = len(sampled_indices)

    labels = np.zeros(n_sampled, dtype=np.int64)
    action_segs = sorted(
        [a["segment"] for a in info["annotations"] if a["label"] == ACTION],
        key=lambda s: s[0],
    )
    for seg_start, seg_end in action_segs:
        for i, fi in enumerate(sampled_indices):
            if seg_start <= fi / fps <= seg_end:
                labels[i] = 1

    if len(action_segs) == 0:
        bags.append((vid, sampled_indices, 0, labels))
        continue

    diff = np.diff(labels, prepend=0, append=0)
    starts = np.where(diff == 1)[0]
    ends   = np.where(diff == -1)[0]

    boundaries = [0]
    for i in range(len(starts) - 1):
        mid = (ends[i] + starts[i + 1]) // 2
        boundaries.append(mid)
    boundaries.append(n_sampled)

    for i in range(len(boundaries) - 1):
        s, e = boundaries[i], boundaries[i + 1]
        seg_labels = labels[s:e]
        bags.append((vid, sampled_indices[s:e], int(seg_labels.max()), seg_labels))

print(f"Total bags: {len(bags)}")
print(f"Positive: {sum(b[2] for b in bags)}, Negative: {sum(1-b[2] for b in bags)}")

Total bags: 1038
Positive: 673, Negative: 365


## 2. Load ViT-B/16 encoder

In [7]:
weights = ViT_B_16_Weights.DEFAULT
model = vit_b_16(weights=weights)
model.heads = torch.nn.Identity()  # remove classification head → 768-d embeddings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

preprocess = weights.transforms()
print(f"Device: {device}, embedding dim: 768")

Device: cpu, embedding dim: 768


## 3. Encode each bag

In [8]:
all_X, all_lengths, all_bag_labels, all_instance_labels = [], [], [], []

for idx, (vid, frame_indices, bag_label, inst_labels) in enumerate(bags):
    video_path = os.path.join(VIDEO_DIR, f"{vid}.mp4")

    # read full video once per video (returns T,H,W,C uint8)
    if idx == 0 or bags[idx - 1][0] != vid:
        video_frames = read_video_frames(video_path)

    # select sampled frames
    valid = [fi for fi in frame_indices if fi < len(video_frames)]
    frames = video_frames[valid]  # (N, H, W, C)
    frames = frames.permute(0, 3, 1, 2).float() / 255.0  # (N, C, H, W)

    # preprocess & encode in batches of 16
    embeddings = []
    for i in range(0, len(frames), 16):
        batch = torch.stack([preprocess(f) for f in frames[i:i+16]])
        with torch.no_grad():
            emb = model(batch.to(device)).cpu()
        embeddings.append(emb)

    if len(embeddings) == 0:
        continue

    embeddings = torch.cat(embeddings)  # (N, 768)
    trimmed_labels = inst_labels[:len(valid)]

    all_X.append(embeddings)
    all_lengths.append(len(embeddings))
    all_bag_labels.append(bag_label)
    all_instance_labels.append(torch.tensor(trimmed_labels, dtype=torch.long))

    if (idx + 1) % 50 == 0:
        print(f"Encoded {idx+1}/{len(bags)} bags")

print(f"Done. {len(all_X)} bags encoded.")

Encoded 50/1038 bags
Encoded 100/1038 bags


KeyboardInterrupt: 

## 4. Save

In [ ]:
torch.save({
    "X": torch.cat(all_X),                                    # (total_frames, 768)
    "lengths": tuple(all_lengths),                             # per-bag frame counts
    "bag_labels": torch.tensor(all_bag_labels, dtype=torch.long),  # (n_bags,)
    "instance_labels": all_instance_labels,                    # list of (L_i,) tensors
}, os.path.join(OUT_DIR, "thumos_diving.pth"))

print(f"Saved to {OUT_DIR}/thumos_diving.pth")
print(f"Bags: {len(all_lengths)}, Total frames: {sum(all_lengths)}")
print(f"Positive bags: {sum(all_bag_labels)}, Negative bags: {len(all_bag_labels)-sum(all_bag_labels)}")